In [ ]:
!pip install rdkit wandb

In [ ]:
import os
if not os.path.exists('src'):
    os.makedirs('src')
if not os.path.exists('sweep'):
    os.makedirs('sweep')

In [ ]:
%%writefile src/utils.py
import math
from torch.nn.init import _calculate_fan_in_and_fan_out, _no_grad_normal_, _no_grad_uniform_

def earily_stop(val_acc_history, tasks, early_stop_step_single,
                early_stop_step_multi, required_progress):
    if len(tasks) == 1:
        t = early_stop_step_single
    else:
        t = early_stop_step_multi

    if len(val_acc_history)>t:
        if val_acc_history[-1] - val_acc_history[-1-t] < required_progress:
            return True
    return False

def xavier_normal_small_init_(tensor, gain=1.):
    fan_in, fan_out = _calculate_fan_in_and_fan_out(tensor)
    std = gain * math.sqrt(2.0 / float(fan_in + 4*fan_out))
    return _no_grad_normal_(tensor, 0., std)

def xavier_uniform_small_init_(tensor, gain=1.):
    fan_in, fan_out = _calculate_fan_in_and_fan_out(tensor)
    std = gain * math.sqrt(2.0 / float(fan_in + 4*fan_out))
    a = math.sqrt(3.0) * std
    return _no_grad_uniform_(tensor, -a, a)

In [ ]:
%%writefile src/transformer.py
import math, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
from utils import xavier_normal_small_init_, xavier_uniform_small_init_

def make_model(d_atom, N=2, d_model=128, h=8, dropout=0.1,
               lambda_attention=0.3, lambda_distance=0.3, trainable_lambda=False,
               N_dense=2, leaky_relu_slope=0.0, aggregation_type='mean',
               dense_output_nonlinearity='relu', distance_matrix_kernel='softmax',
               use_edge_features=False, n_output=1,
               control_edges=False, integrated_distances=False,
               scale_norm=False, init_type='uniform', use_adapter=False, n_generator_layers=1):
    c = copy.deepcopy
    attn = MultiHeadedAttention(h, d_model, dropout, lambda_attention, lambda_distance, trainable_lambda, distance_matrix_kernel, use_edge_features, control_edges, integrated_distances)
    ff = PositionwiseFeedForward(d_model, N_dense, dropout, leaky_relu_slope, dense_output_nonlinearity)
    model = GraphTransformer(
        Encoder(EncoderLayer(d_model, c(attn), c(ff), dropout, scale_norm, use_adapter), N, scale_norm),
        Embeddings(d_model, d_atom, dropout),
        Generator(d_model, aggregation_type, n_output, n_generator_layers, leaky_relu_slope, dropout, scale_norm))

    for p in model.parameters():
        if p.dim() > 1:
            if init_type == 'uniform':
                nn.init.xavier_uniform_(p)
            elif init_type == 'normal':
                nn.init.xavier_normal_(p)
            elif init_type == 'small_normal_init':
                xavier_normal_small_init_(p)
            elif init_type == 'small_uniform_init':
                xavier_uniform_small_init_(p)
    return model

class GraphTransformer(nn.Module):
    def __init__(self, encoder, src_embed, generator):
        super(GraphTransformer, self).__init__()
        self.encoder = encoder
        self.src_embed = src_embed
        self.generator = generator

    def forward(self, src, src_mask, adj_matrix, distances_matrix, edges_att):
        return self.predict(self.encode(src, src_mask, adj_matrix, distances_matrix, edges_att), src_mask)

    def encode(self, src, src_mask, adj_matrix, distances_matrix, edges_att):
        return self.encoder(self.src_embed(src), src_mask, adj_matrix, distances_matrix, edges_att)

    def predict(self, out, out_mask):
        return self.generator(out, out_mask)

class Generator(nn.Module):
    def __init__(self, d_model, aggregation_type='mean', n_output=1, n_layers=1,
                 leaky_relu_slope=0.01, dropout=0.0, scale_norm=False):
        super(Generator, self).__init__()
        if n_layers == 1:
            self.proj = nn.Linear(d_model, n_output)
        else:
            self.proj = []
            for i in range(n_layers-1):
                self.proj.append(nn.Linear(d_model, d_model))
                self.proj.append(nn.LeakyReLU(leaky_relu_slope))
                self.proj.append(ScaleNorm(d_model) if scale_norm else LayerNorm(d_model))
                self.proj.append(nn.Dropout(dropout))
            self.proj.append(nn.Linear(d_model, n_output))
            self.proj = torch.nn.Sequential(*self.proj)
        self.aggregation_type = aggregation_type

    def forward(self, x, mask):
        mask = mask.unsqueeze(-1).float()
        out_masked = x * mask
        if self.aggregation_type == 'mean':
            out_sum = out_masked.sum(dim=1)
            mask_sum = mask.sum(dim=(1))
            out_avg_pooling = out_sum / mask_sum
        elif self.aggregation_type == 'sum':
            out_sum = out_masked.sum(dim=1)
            out_avg_pooling = out_sum
        elif self.aggregation_type == 'dummy_node':
            out_avg_pooling = out_masked[:,0]
        projected = self.proj(out_avg_pooling)
        return projected

class PositionGenerator(nn.Module):
    def __init__(self, d_model):
        super(PositionGenerator, self).__init__()
        self.norm = LayerNorm(d_model)
        self.proj = nn.Linear(d_model, 3)

    def forward(self, x, mask):
        mask = mask.unsqueeze(-1).float()
        out_masked = self.norm(x) * mask
        projected = self.proj(out_masked)
        return projected

def clones(module, N):
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

class Encoder(nn.Module):
    def __init__(self, layer, N, scale_norm):
        super(Encoder, self).__init__()
        self.layers = clones(layer, N)
        self.norm = ScaleNorm(layer.size) if scale_norm else LayerNorm(layer.size)

    def forward(self, x, mask, adj_matrix, distances_matrix, edges_att):
        for layer in self.layers:
            x = layer(x, mask, adj_matrix, distances_matrix, edges_att)
        return self.norm(x)

class LayerNorm(nn.Module):
    def __init__(self, features, eps=1e-6):
        super(LayerNorm, self).__init__()
        self.a_2 = nn.Parameter(torch.ones(features))
        self.b_2 = nn.Parameter(torch.zeros(features))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return self.a_2 * (x - mean) / (std + self.eps) + self.b_2

class ScaleNorm(nn.Module):
    def __init__(self, scale, eps=1e-5):
        super(ScaleNorm, self).__init__()
        self.scale = nn.Parameter(torch.tensor(math.sqrt(scale)))
        self.eps = eps

    def forward(self, x):
        norm = self.scale / torch.norm(x, dim=-1, keepdim=True).clamp(min=self.eps)
        return x * norm

class SublayerConnection(nn.Module):
    def __init__(self, size, dropout, scale_norm, use_adapter):
        super(SublayerConnection, self).__init__()
        self.norm = ScaleNorm(size) if scale_norm else LayerNorm(size)
        self.dropout = nn.Dropout(dropout)
        self.use_adapter = use_adapter
        self.adapter = Adapter(size, 8) if use_adapter else None

    def forward(self, x, sublayer):
        if self.use_adapter:
            return x + self.dropout(self.adapter(sublayer(self.norm(x))))
        return x + self.dropout(sublayer(self.norm(x)))

class EncoderLayer(nn.Module):
    def __init__(self, size, self_attn, feed_forward, dropout, scale_norm, use_adapter):
        super(EncoderLayer, self).__init__()
        self.self_attn = self_attn
        self.feed_forward = feed_forward
        self.sublayer = clones(SublayerConnection(size, dropout, scale_norm, use_adapter), 2)
        self.size = size

    def forward(self, x, mask, adj_matrix, distances_matrix, edges_att):
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, adj_matrix, distances_matrix, edges_att, mask))
        return self.sublayer[1](x, self.feed_forward)

class EdgeFeaturesLayer(nn.Module):
    def __init__(self, d_model, d_edge, h, dropout):
        super(EdgeFeaturesLayer, self).__init__()
        assert d_model % h == 0
        d_k = d_model // h
        self.linear = nn.Linear(d_edge, 1, bias=False)
        with torch.no_grad():
            self.linear.weight.fill_(0.25)

    def forward(self, x):
        p_edge = x.permute(0, 2, 3, 1)
        p_edge = self.linear(p_edge).permute(0, 3, 1, 2)
        return torch.relu(p_edge)

def attention(query, key, value, adj_matrix, distances_matrix, edges_att,
              mask=None, dropout=None,
              lambdas=(0.3, 0.3, 0.4), trainable_lambda=False,
              distance_matrix_kernel=None, use_edge_features=False, control_edges=False,
              eps=1e-6, inf=1e12):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) \
             / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask.unsqueeze(1).repeat(1, query.shape[1], query.shape[2], 1) == 0, -inf)
    p_attn = F.softmax(scores, dim = -1)

    if use_edge_features:
        adj_matrix = edges_att.view(adj_matrix.shape)

    adj_matrix = adj_matrix / (adj_matrix.sum(dim=-1).unsqueeze(2) + eps)
    adj_matrix = adj_matrix.unsqueeze(1).repeat(1, query.shape[1], 1, 1)
    p_adj = adj_matrix
    p_dist = distances_matrix

    if trainable_lambda:
        softmax_attention, softmax_distance, softmax_adjacency = lambdas.cuda()
        p_weighted = softmax_attention * p_attn + softmax_distance * p_dist + softmax_adjacency * p_adj
    else:
        lambda_attention, lambda_distance, lambda_adjacency = lambdas
        p_weighted = lambda_attention * p_attn + lambda_distance * p_dist + lambda_adjacency * p_adj

    if dropout is not None:
        p_weighted = dropout(p_weighted)

    atoms_featrues = torch.matmul(p_weighted, value)
    return atoms_featrues, p_weighted, p_attn

class MultiHeadedAttention(nn.Module):
    def __init__(self, h, d_model, dropout=0.1, lambda_attention=0.3, lambda_distance=0.3, trainable_lambda=False,
                 distance_matrix_kernel='softmax', use_edge_features=False, control_edges=False, integrated_distances=False):
        super(MultiHeadedAttention, self).__init__()
        assert d_model % h == 0
        self.d_k = d_model // h
        self.h = h
        self.trainable_lambda = trainable_lambda
        if trainable_lambda:
            lambda_adjacency = 1. - lambda_attention - lambda_distance
            lambdas_tensor = torch.tensor([lambda_attention, lambda_distance, lambda_adjacency], requires_grad=True)
            self.lambdas = torch.nn.Parameter(lambdas_tensor)
        else:
            lambda_adjacency = 1. - lambda_attention - lambda_distance
            self.lambdas = (lambda_attention, lambda_distance, lambda_adjacency)

        self.linears = clones(nn.Linear(d_model, d_model), 4)
        self.attn = None
        self.dropout = nn.Dropout(p=dropout)
        if distance_matrix_kernel == 'softmax':
            self.distance_matrix_kernel = lambda x: F.softmax(-x, dim = -1)
        elif distance_matrix_kernel == 'exp':
            self.distance_matrix_kernel = lambda x: torch.exp(-x)
        self.integrated_distances = integrated_distances
        self.use_edge_features = use_edge_features
        self.control_edges = control_edges
        if use_edge_features:
            d_edge = 11 if not integrated_distances else 12
            self.edges_feature_layer = EdgeFeaturesLayer(d_model, d_edge, h, dropout)

    def forward(self, query, key, value, adj_matrix, distances_matrix, edges_att, mask=None):
        if mask is not None:
            mask = mask.unsqueeze(1)
        nbatches = query.size(0)
        query, key, value = \
            [l(x).view(nbatches, -1, self.h, self.d_k).transpose(1, 2)
             for l, x in zip(self.linears, (query, key, value))]
        distances_matrix = distances_matrix.masked_fill(mask.repeat(1, mask.shape[-1], 1) == 0, np.inf)
        distances_matrix = self.distance_matrix_kernel(distances_matrix)
        p_dist = distances_matrix.unsqueeze(1).repeat(1, query.shape[1], 1, 1)

        if self.use_edge_features:
            if self.integrated_distances:
                edges_att = torch.cat((edges_att, distances_matrix.unsqueeze(1)), dim=1)
            edges_att = self.edges_feature_layer(edges_att)

        x, self.attn, self.self_attn = attention(query, key, value, adj_matrix,
                                                 p_dist, edges_att,
                                                 mask=mask, dropout=self.dropout,
                                                 lambdas=self.lambdas,
                                                 trainable_lambda=self.trainable_lambda,
                                                 distance_matrix_kernel=self.distance_matrix_kernel,
                                                 use_edge_features=self.use_edge_features,
                                                 control_edges=self.control_edges)
        x = x.transpose(1, 2).contiguous() \
             .view(nbatches, -1, self.h * self.d_k)
        return self.linears[-1](x)

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, N_dense, dropout=0.1, leaky_relu_slope=0.0, dense_output_nonlinearity='relu'):
        super(PositionwiseFeedForward, self).__init__()
        self.N_dense = N_dense
        self.linears = clones(nn.Linear(d_model, d_model), N_dense)
        self.dropout = clones(nn.Dropout(dropout), N_dense)
        self.leaky_relu_slope = leaky_relu_slope
        if dense_output_nonlinearity == 'relu':
            self.dense_output_nonlinearity = lambda x: F.leaky_relu(x, negative_slope=self.leaky_relu_slope)
        elif dense_output_nonlinearity == 'tanh':
            self.tanh = torch.nn.Tanh()
            self.dense_output_nonlinearity = lambda x: self.tanh(x)
        elif dense_output_nonlinearity == 'none':
            self.dense_output_nonlinearity = lambda x: x

    def forward(self, x):
        if self.N_dense == 0:
            return x
        for i in range(len(self.linears)-1):
            x = self.dropout[i](F.leaky_relu(self.linears[i](x), negative_slope=self.leaky_relu_slope))
        return self.dropout[-1](self.dense_output_nonlinearity(self.linears[-1](x)))

class Embeddings(nn.Module):
    def __init__(self, d_model, d_atom, dropout):
        super(Embeddings, self).__init__()
        self.lut = nn.Linear(d_atom, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.lut(x))

In [ ]:
%%writefile src/data_utils.py
import logging
import os
import pickle
import numpy as np
import pandas as pd
import torch
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import MolFromSmiles
from sklearn.metrics import pairwise_distances
from torch.utils.data import Dataset

use_cuda = torch.cuda.is_available()
FloatTensor = torch.cuda.FloatTensor if use_cuda else torch.FloatTensor
LongTensor = torch.cuda.LongTensor if use_cuda else torch.LongTensor
IntTensor = torch.cuda.IntTensor if use_cuda else torch.IntTensor
DoubleTensor = torch.cuda.DoubleTensor if use_cuda else torch.DoubleTensor

def load_data_from_df(dataset_path, add_dummy_node=True, one_hot_formal_charge=False, use_data_saving=True,two_d_only=False):
    feat_stamp = f'{"_dn" if add_dummy_node else ""}{"_ohfc" if one_hot_formal_charge else ""}'
    feature_path = dataset_path.replace('.csv', f'{feat_stamp}.p')
    if use_data_saving and os.path.exists(feature_path):
        logging.info(f"Loading features stored at '{feature_path}'")
        x_all, y_all = pickle.load(open(feature_path, "rb"))
        return x_all, y_all

    data_df = pd.read_csv(dataset_path)
    data_x = data_df.iloc[:, 0].values
    data_y = data_df.iloc[:, 1].values

    if data_y.dtype == np.float64:
        data_y = data_y.astype(np.float32)

    x_all, y_all = load_data_from_smiles(data_x, data_y, add_dummy_node=add_dummy_node,
                                         one_hot_formal_charge=one_hot_formal_charge,two_d_only=two_d_only)
    if use_data_saving and not os.path.exists(feature_path):
        logging.info(f"Saving features at '{feature_path}'")
        pickle.dump((x_all, y_all), open(feature_path, "wb"))

    return x_all, y_all

def load_data_from_smiles(x_smiles, labels, add_dummy_node=True, one_hot_formal_charge=False,two_d_only=False):
    x_all, y_all = [], []
    for smiles, label in zip(x_smiles, labels):
        try:
            mol = MolFromSmiles(smiles)
            if two_d_only:
                AllChem.Compute2DCoords(mol)
            else:
                try:
                    mol = Chem.AddHs(mol)
                    AllChem.EmbedMolecule(mol, maxAttempts=5000)
                    AllChem.UFFOptimizeMolecule(mol)
                    mol = Chem.RemoveHs(mol)
                except:
                    AllChem.Compute2DCoords(mol)

            afm, adj, dist = featurize_mol(mol, add_dummy_node, one_hot_formal_charge)
            x_all.append([afm, adj, dist])
            y_all.append([label])
        except ValueError as e:
            logging.warning('the SMILES ({}) can not be converted to a graph.\nREASON: {}'.format(smiles, e))
    return x_all, y_all

def featurize_mol(mol, add_dummy_node, one_hot_formal_charge):
    node_features = np.array([get_atom_features(atom, one_hot_formal_charge)
                              for atom in mol.GetAtoms()])

    adj_matrix = np.eye(mol.GetNumAtoms())
    for bond in mol.GetBonds():
        begin_atom = bond.GetBeginAtom().GetIdx()
        end_atom = bond.GetEndAtom().GetIdx()
        adj_matrix[begin_atom, end_atom] = adj_matrix[end_atom, begin_atom] = 1

    conf = mol.GetConformer()
    pos_matrix = np.array([[conf.GetAtomPosition(k).x, conf.GetAtomPosition(k).y, conf.GetAtomPosition(k).z]
                           for k in range(mol.GetNumAtoms())])
    dist_matrix = pairwise_distances(pos_matrix)

    if add_dummy_node:
        m = np.zeros((node_features.shape[0] + 1, node_features.shape[1] + 1))
        m[1:, 1:] = node_features
        m[0, 0] = 1.
        node_features = m

        m = np.zeros((adj_matrix.shape[0] + 1, adj_matrix.shape[1] + 1))
        m[1:, 1:] = adj_matrix
        adj_matrix = m

        m = np.full((dist_matrix.shape[0] + 1, dist_matrix.shape[1] + 1), 1e6)
        m[1:, 1:] = dist_matrix
        dist_matrix = m

    return node_features, adj_matrix, dist_matrix

def get_atom_features(atom, one_hot_formal_charge=True):
    attributes = []
    attributes += one_hot_vector(atom.GetAtomicNum(), [5, 6, 7, 8, 9, 15, 16, 17, 35, 53, 999])
    attributes += one_hot_vector(len(atom.GetNeighbors()), [0, 1, 2, 3, 4, 5])
    attributes += one_hot_vector(atom.GetTotalNumHs(), [0, 1, 2, 3, 4])
    if one_hot_formal_charge:
        attributes += one_hot_vector(atom.GetFormalCharge(), [-1, 0, 1])
    else:
        attributes.append(atom.GetFormalCharge())
    attributes.append(atom.IsInRing())
    attributes.append(atom.GetIsAromatic())
    return np.array(attributes, dtype=np.float32)

def one_hot_vector(val, lst):
    if val not in lst:
        val = lst[-1]
    return map(lambda x: x == val, lst)

class Molecule:
    def __init__(self, x, y, index):
        self.node_features = x[0]
        self.adjacency_matrix = x[1]
        self.distance_matrix = x[2]
        self.y = y
        self.index = index

class MolDataset(Dataset):
    def __init__(self, data_list):
        self.data_list = data_list
    def __len__(self):
        return len(self.data_list)
    def __getitem__(self, key):
        if type(key) == slice:
            return MolDataset(self.data_list[key])
        return self.data_list[key]

def pad_array(array, shape, dtype=np.float32):
    padded_array = np.zeros(shape, dtype=dtype)
    padded_array[:array.shape[0], :array.shape[1]] = array
    return padded_array

def mol_collate_func(batch):
    adjacency_list, distance_list, features_list, labels = [], [], [], []
    max_size = 0
    for molecule in batch:
        if type(molecule.y[0]) == np.ndarray:
            labels.append(molecule.y[0])
        else:
            labels.append(molecule.y)
        if molecule.adjacency_matrix.shape[0] > max_size:
            max_size = molecule.adjacency_matrix.shape[0]

    for molecule in batch:
        adjacency_list.append(pad_array(molecule.adjacency_matrix, (max_size, max_size)))
        distance_list.append(pad_array(molecule.distance_matrix, (max_size, max_size)))
        features_list.append(pad_array(molecule.node_features, (max_size, molecule.node_features.shape[1])))

    return [FloatTensor(np.array(features)) for features in (adjacency_list, features_list, distance_list, labels)]

def construct_dataset(x_all, y_all):
    output = [Molecule(data[0], data[1], i) for i, data in enumerate(zip(x_all, y_all))]
    return MolDataset(output)

def construct_loader(x, y, batch_size, shuffle=True):
    data_set = construct_dataset(x, y)
    loader = torch.utils.data.DataLoader(dataset=data_set, batch_size=batch_size, collate_fn=mol_collate_func, shuffle=shuffle)
    return loader

In [ ]:
%%writefile train.py
#!/usr/bin/env python3
import os
import sys
import pandas as pd
import torch
import numpy as np
import wandb
import time
sys.path.append(os.getcwd())
sys.path.append('src')
import argparse
from transformer import make_model
import pickle

parser=argparse.ArgumentParser(description='Train and test MAT model')
parser.add_argument('--trainfile',type=str,default='',help='Specify training data file')
parser.add_argument('--testfile',type=str,default='',help='Spefify testing data file')
parser.add_argument('--prefix',type=str,default='',help='Prefix for the train and test data')
parser.add_argument('--fold',type=str,default='',help='Fold for the datafiles')
parser.add_argument('--datadir',type=str,default='sweep',help='Absolutepath to the directory for the data')
parser.add_argument('--savemodel', action='store_true',default=False,help='Flag to save the trained model')
parser.add_argument('--dynamic',default=0,type=int,help='Early stopping epochs')
parser.add_argument('-e','--epochs', type=int, default=500,help='Number of epochs')
parser.add_argument('-l','--loss',type=str,default='huber',help='Loss Function')
parser.add_argument('-o','--optimizer',type=str,default='sgd',help='Optimizer')
parser.add_argument('--lr',type=float,default=1e-4, help='Learning rate')
parser.add_argument('--momentum',type=float,default=0.9,help='Momentum for SGD')
parser.add_argument('--weight_decay',type=float,default=0,help='L2 pentalty')
parser.add_argument('--dampening',type=float,default=0,help='Dampening for momentum')
parser.add_argument('--beta1',type=float,default=0.9,help='Beta1 for ADAM')
parser.add_argument('--beta2',type=float,default=0.999,help='Beta2 for ADAM')
parser.add_argument('--epsilon',type=float,default=1e-08,help='Epsilon for ADAM')
parser.add_argument('--dropout',type=float,default=0,help='Dropout')
parser.add_argument('--ldist',type=float,default=0.33,help='Lambda for model attention to the distance matrix')
parser.add_argument('--lattn',type=float,default=0.33,help='Lambda for model attention to the attention matrix')
parser.add_argument('--Ndense',type=int,default=1,help='Number of Dense blocks')
parser.add_argument('--heads',type=int,default=16,help='Number of attention heads')
parser.add_argument('--dmodel',type=int,default=1024,help='Dimension of the hidden layer')
parser.add_argument('--nstacklayers',type=int,default=8,help='Number of stacks')
parser.add_argument('--cpu',action='store_true',default=False,help='Flag to have model be CPU only')
parser.add_argument('--wandb',default=None,help='Project name for weights and biases')
parser.add_argument('--twod',action='store_true',help='Flag to only use 2D conformers')
parser.add_argument('--seed',type=int,default=420,help='Random seed')

args=parser.parse_args()

if args.cpu:
    from cpu_data_utils import load_data_from_df, construct_loader
else:
    from data_utils import load_data_from_df, construct_loader
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False

assert args.loss in set(['mse','mae','huber','logcosh']) and args.optimizer in set(['sgd','adam'])

if args.trainfile:
    trainfile=args.trainfile
    testfile=args.testfile
    if args.cpu:
        outf_prefix=f'aqsol_test_ind_cpu_drop{args.dropout}_ldist{args.ldist}_lattn{args.lattn}_Ndense{args.Ndense}_heads{args.heads}_dmodel{args.dmodel}_nsl{args.nstacklayers}_epochs{args.epochs}_dyn{args.dynamic}_seed{args.seed}'
    else:
        outf_prefix=f'aqsol_test_ind_drop{args.dropout}_ldist{args.ldist}_lattn{args.lattn}_Ndense{args.Ndense}_heads{args.heads}_dmodel{args.dmodel}_nsl{args.nstacklayers}_epochs{args.epochs}_dyn{args.dynamic}_seed{args.seed}'
else:
    trainfile=args.prefix+'_train'+args.fold+'.csv'
    testfile=args.prefix+'_test'+args.fold+'.csv'
    namep=args.prefix.split('/')[-1]
    if args.cpu:
        outf_prefix=f'{namep}_cpu_{args.fold}_drop{args.dropout}_ldist{args.ldist}_lattn{args.lattn}_Ndense{args.Ndense}_heads{args.heads}_dmodel{args.dmodel}_nsl{args.nstacklayers}_epochs{args.epochs}_dyn{args.dynamic}_seed{args.seed}'
    else:
        outf_prefix=f'{namep}_{args.fold}_drop{args.dropout}_ldist{args.ldist}_lattn{args.lattn}_Ndense{args.Ndense}_heads{args.heads}_dmodel{args.dmodel}_nsl{args.nstacklayers}_epochs{args.epochs}_dyn{args.dynamic}_seed{args.seed}'

if args.twod and '2d' not in args.prefix and '2d' not in args.trainfile:
    outf_prefix=outf_prefix.replace('_drop','_2d_drop')

if args.wandb:
    wandb.init(project=args.wandb,name=outf_prefix)

print('Trainfile:',trainfile)
print('Testfile:',testfile)
print('Outfile Prefix:',outf_prefix)
print('Loading train and test data')

torch.manual_seed(args.seed)
np.random.seed(args.seed)

batch_size=64
trainX, trainy=load_data_from_df(trainfile,one_hot_formal_charge=True,two_d_only=args.twod)
data_loader=construct_loader(trainX,trainy,batch_size)
testX, testy=load_data_from_df(testfile,one_hot_formal_charge=True,two_d_only=args.twod)
testdata_loader=construct_loader(testX,testy,batch_size)

d_atom = trainX[0][0].shape[1]

model_params= {
    'd_atom': d_atom,
    'd_model': args.dmodel,
    'N': args.nstacklayers,
    'h': args.heads,
    'N_dense': args.Ndense,
    'lambda_attention': args.lattn,
    'lambda_distance': args.ldist,
    'leaky_relu_slope': 0.1,
    'dense_output_nonlinearity': 'relu',
    'distance_matrix_kernel': 'exp',
    'dropout': args.dropout,
    'aggregation_type': 'mean'
}

print('Making Model')

model=make_model(**model_params)
param_count=sum(p.numel() for p in model.parameters() if p.requires_grad)
print('Number of parameters:',param_count)

if args.wandb:
    wandb.watch(model,'all')
    wandb.log({'Parameters':param_count},step=0)

if args.cpu:
    model.cpu()
else:
    torch.cuda.empty_cache()
    model.cuda()
model.train()

if args.loss == 'mse':
    criterion=torch.nn.MSELoss(reduction='mean')
elif args.loss=='mae':
    criterion=torch.nn.L1Loss(reduction='mean')
elif args.loss=='huber':
    criterion=torch.nn.SmoothL1Loss(reduction='mean')

if args.optimizer=='sgd':
    optimizer=torch.optim.SGD(model.parameters(),lr=args.lr,momentum=args.momentum,weight_decay=args.weight_decay)
elif args.optimizer=='adam':
    optimizer=torch.optim.Adam(model.parameters(),lr=args.lr,weight_decay=args.weight_decay)

losses=[]
num_epochs_since_improvement=0
last_test_rmse=999999
last_test_r2=-1

print('Training')
iteration=0
for epoch in range(args.epochs):
    epoch_preds=np.array([])
    epoch_gold=np.array([])
    for batch in data_loader:
        iteration+=1
        optimizer.zero_grad()
        adjacency_matrix, node_features, distance_matrix, y = batch
        batch_mask = torch.sum(torch.abs(node_features), dim=-1) != 0
        y_pred = model(node_features, batch_mask, adjacency_matrix, distance_matrix, None)

        epoch_gold=np.append(epoch_gold,y.tolist())
        epoch_preds=np.append(epoch_preds,y_pred.tolist())

        loss=criterion(y_pred,y)
        losses.append(loss.item())
        if args.wandb:
            wandb.log({"Train Loss":loss.item()},step=iteration)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2)
        optimizer.step()

        if not args.cpu:
            torch.cuda.empty_cache()

        if iteration%1==0:
            model.eval()
            gold=np.array([])
            preds=np.array([])
            for t_batch in testdata_loader:
                t_adjacency_matrix, t_node_features, t_distance_matrix, t_y = t_batch
                gold=np.append(gold,t_y.tolist())
                t_batch_mask = torch.sum(torch.abs(t_node_features), dim=-1) != 0
                t_y_pred = model(t_node_features, t_batch_mask, t_adjacency_matrix, t_distance_matrix, None)
                preds=np.append(preds,t_y_pred.tolist())
                if not args.cpu:
                    torch.cuda.empty_cache()
            test_rmse=np.sqrt(np.mean((preds-gold)**2))
            test_r2=np.corrcoef(preds,gold)[0][1]**2
            model.train()
            if args.wandb:
                wandb.log({"Test RMSE":test_rmse,"Test R2":test_r2},step=iteration)

    train_rmse, train_r2=(np.sqrt(np.mean((epoch_preds-epoch_gold)**2)), np.corrcoef(epoch_preds,epoch_gold)[0][1]**2)
    if args.wandb:
        wandb.log({"Train Epoch RMSE":train_rmse,"Train Epoch R2":train_r2},step=iteration)

    if args.dynamic>0:
        model.eval()
        gold=np.array([])
        preds=np.array([])
        for t_batch in testdata_loader:
            t_adjacency_matrix, t_node_features, t_distance_matrix, t_y = t_batch
            gold=np.append(gold,t_y.tolist())
            t_batch_mask = torch.sum(torch.abs(t_node_features), dim=-1) != 0
            t_y_pred = model(t_node_features, t_batch_mask, t_adjacency_matrix, t_distance_matrix, None)
            preds=np.append(preds,t_y_pred.tolist())
            if not args.cpu:
                torch.cuda.empty_cache()
        test_rmse=np.sqrt(np.mean((preds-gold)**2))
        test_r2=np.corrcoef(preds,gold)[0][1]**2
        model.train()

        if test_r2 > last_test_r2 or test_rmse < last_test_rmse:
            last_test_rmse=test_rmse
            last_test_r2=test_r2
            num_epochs_since_improvement=0
        else:
            num_epochs_since_improvement+=1

        print(f'----------------------------------')
        print(f'Epoch: {epoch}/{args.epochs}')
        print(f'Training RMSE: {train_rmse}')
        print(f'Training R2: {train_r2}')
        print(f'Test RMSE: {test_rmse}')
        print(f'Test R2: {test_r2}')
        print(f'Num since Improvement: {num_epochs_since_improvement}')
        print(f'Best RMSE: {last_test_rmse}')
        print(f'Best R2: {last_test_r2}')
        print(f'----------------------------------')
        if num_epochs_since_improvement==args.dynamic:
            print(f'Early termination signalled! Stopping training!\n{epoch}/{args.epochs} Completed.')
            break

print('Training Complete!')
if args.savemodel:
    print('Saving Model:',args.datadir+'/'+outf_prefix+'_trained.model')
    torch.save(model.state_dict(),args.datadir+'/'+outf_prefix+'_trained.model')

if not os.path.isdir(args.datadir):
    os.mkdir(args.datadir)

with open(args.datadir+'/'+outf_prefix+'_trainlosses.pi','wb') as outfile:
    pickle.dump(losses,outfile)

print('Final Evaluations!')
print('Training Set:')
model.eval()

gold=np.array([])
preds=np.array([])
t0=time.time()
train_times=[]
for batch in data_loader:
    t1=time.time()
    adjacency_matrix, node_features, distance_matrix, y = batch
    tload=time.time()-t1
    gold=np.append(gold,y.tolist())
    batch_mask = torch.sum(torch.abs(node_features), dim=-1) != 0
    y_pred = model(node_features, batch_mask, adjacency_matrix, distance_matrix, None)
    tpred=time.time()-t1
    preds=np.append(preds,y_pred.tolist())
    if not args.cpu:
        torch.cuda.empty_cache()
    train_times.append((tload,tpred))
ttime=time.time()-t0
print('Overall Time: ',ttime)
if args.wandb:
    wandb.log({'Train Eval Time':ttime},step=iteration+1)
r2=np.corrcoef(preds,gold)[0][1]**2
rmse=np.sqrt(np.mean((preds-gold)**2))
if args.wandb:
    wandb.log({"Train Epoch RMSE":rmse,"Train Epoch R2":r2},step=iteration+1)

print('Test Set:')
gold=np.array([])
preds=np.array([])
t0=time.time()
test_times=[]
for batch in testdata_loader:
    t1=time.time()
    adjacency_matrix, node_features, distance_matrix, y = batch
    tload=time.time()-t1
    gold=np.append(gold,y.tolist())
    batch_mask = torch.sum(torch.abs(node_features), dim=-1) != 0
    y_pred = model(node_features, batch_mask, adjacency_matrix, distance_matrix, None)
    tpred=time.time()-t1
    preds=np.append(preds,y_pred.tolist())
    if not args.cpu:
        torch.cuda.empty_cache()
    test_times.append((tload,tpred))
ttime=time.time()-t0
print('Overall Time: ',ttime)
if args.wandb:
    wandb.log({'Test Eval Time':ttime},step=iteration+1)

r2=np.corrcoef(preds,gold)[0][1]**2
rmse=np.sqrt(np.mean((preds-gold)**2))

testdic={
    'predicted':preds,
    'target':gold,
    'RMSE':rmse,
    'R2':r2,
    'Traintimes':train_times,
    'Testtimes':test_times
}
print('Test RMSE:',rmse)
print('Test R2  :',r2)

if args.wandb:
    wandb.log({"Test RMSE":rmse,"Test R2":r2},step=iteration+1)

with open(args.datadir+'/'+outf_prefix+'_testdic.pi','wb') as outfile:
    pickle.dump(testdic,outfile)

In [ ]:
%%writefile predict.py
#!/usr/bin/env python3

import os
import sys
import pandas as pd
import torch
import numpy as np

sys.path.append(os.getcwd())
sys.path.append('src')

import argparse
import time
from transformer import make_model
from data_utils import load_data_from_df, construct_loader
import pickle
import re

def parse_model_options(filename):
    drop=float(re.search(r'drop(\d+\.?\d*)',filename).group(1))
    ldist=float(re.search(r'ldist(\d+\.?\d*)',filename).group(1))
    lattn=float(re.search(r'lattn(\d+\.?\d*)',filename).group(1))
    nDense=int(re.search(r'Ndense(\d+\.?\d*)',filename).group(1))
    heads=int(re.search(r'heads(\d+\.?\d*)',filename).group(1))
    dmodel=int(re.search(r'dmodel(\d+\.?\d*)',filename).group(1))
    nsl=int(re.search(r'nsl(\d+\.?\d*)',filename).group(1))
    return drop,ldist,lattn,nDense,heads,dmodel,nsl

parser=argparse.ArgumentParser(description='Predict MAT model on a given test set')
parser.add_argument('-m','--model',type=str,required=True,help='Trained torch model file')
parser.add_argument('-i','--input',type=str,required=True,help='File to evaluate. Assumed format is "<SMILE>,<solubility>"')
parser.add_argument('-o','--output',type=str,required=True,help='File for Predictions. Format is "<SMILE>,<True>,<Predicted>"')
parser.add_argument('--stats',default=False,action='store_true',help='Flag to print the R2, RMSE, and the time to perform the evaluation.')
parser.add_argument('--twod',default=False,action='store_true',help='Flag to use 2D conformers for distance matrix.')
args=parser.parse_args()

if args.stats:
    start=time.time()

X,gold=load_data_from_df(args.input,one_hot_formal_charge=True,use_data_saving=True,two_d_only=args.twod)
data_loader=construct_loader(X,gold,batch_size=8,shuffle=False)

if args.stats:
    print('Loading data time:',time.time()-start)
    start=time.time()

drop,ldist,lattn,nDense,heads,dmodel,nsl=parse_model_options(args.model)

d_atom=X[0][0].shape[1]
model_params={
    'd_atom': d_atom,
    'd_model': dmodel,
    'N': nsl,
    'h': heads,
    'N_dense': nDense,
    'lambda_attention': lattn,
    'lambda_distance': ldist,
    'leaky_relu_slope': 0.1,
    'dense_output_nonlinearity': 'relu',
    'distance_matrix_kernel': 'exp',
    'dropout': drop,
    'aggregation_type': 'mean'
}

model=make_model(**model_params)
model.load_state_dict(torch.load(args.model))
model.cuda()
model.eval()

if args.stats:
    print('Model construction time:',time.time()-start)
    start=time.time()

preds=np.array([])
ys=np.array([])
for batch in data_loader:
    adjacency_matrix, node_features,distance_matrix,y=batch
    batch_mask=torch.sum(torch.abs(node_features),dim=-1) != 0
    pred=model(node_features,batch_mask,adjacency_matrix,distance_matrix,None)
    preds=np.append(preds,pred.tolist())
    ys=np.append(ys,y.tolist())

if args.stats:
    print('Model Prediction time:',time.time()-start)
    print('RMSE:',np.sqrt(np.mean((preds-ys)**2)))
    print('R2:',np.corrcoef(preds,ys)[0][1]**2)

with open(args.output,'w') as outfile:
    outfile.write('smile,true,pred\n')
    lines=open(args.input).readlines()
    lines=lines[1:]
    preds=preds.tolist()
    assert len(lines)==len(preds)
    for l,p in zip(lines,preds):
        outfile.write(l.rstrip()+','+str(p)+'\n')

In [ ]:
import pandas as pd

# 1. Load your current files
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 2. Select ONLY the SMILES and LogS columns
# This forces LogS to be the 2nd column, which SolTrannet requires
train_clean = train_df[['SMILES', 'LogS']]
test_clean = test_df[['SMILES', 'LogS']]

# 3. Save as new files
train_clean.to_csv('soltrannet_train.csv', index=False)
test_clean.to_csv('soltrannet_test.csv', index=False)

print("Created soltrannet_train.csv and soltrannet_test.csv")

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

seed = 101

# 1. Load ESOL
print("Loading ESOL dataset...")
esol_url = "https://raw.githubusercontent.com/deepchem/deepchem/master/datasets/delaney-processed.csv"
full_df = pd.read_csv(esol_url)

# 2. Rename columns
full_df = full_df.rename(columns={
    "smiles": "SMILES",
    "measured log solubility in mols per litre": "LogS"
})

# 3. Split (Seed 101)
train_df, test_df = train_test_split(full_df, test_size=0.2, random_state=seed)

# 4. Save with ONLY 2 columns [SMILES, Target] for SolTrannet
# (SolTrannet crashes if there are extra columns or if Target is not the 2nd column)
train_df[['SMILES', 'LogS']].to_csv('esol_soltrannet_train.csv', index=False)
test_df[['SMILES', 'LogS']].to_csv('esol_soltrannet_test.csv', index=False)

print("Created esol_soltrannet_train.csv and esol_soltrannet_test.csv")

In [ ]:
# 1. Clear old cache so it doesn't mix with previous datasets
!rm -f *.p src/*.p

# 2. Run Training
!python train.py \
  --trainfile AqSolDB_curated.csv \
  --testfile sc2.csv \
  --savemodel \
  --epochs 200 \
  --dynamic 10 \
  --optimizer adam \
  --lr 0.0001 \
  --loss mse \
  --dmodel 128 \
  --heads 4 \
  --nstacklayers 4 \
  --dropout 0.1 \
  --ldist 0.33 \
  --lattn 0.33 \
  --Ndense 2 \
  --twod

In [ ]:
!python predict.py \
  --model sweep/aqsol_test_ind_2d_drop0.1_ldist0.33_lattn0.33_Ndense2_heads4_dmodel128_nsl4_epochs200_dyn10_seed420_trained.model \
  --input sc2_clean_for_soltrannet.csv \
  --output sc2_predictions.csv \
  --twod

In [ ]:
import pandas as pd

# 1. Load your SC2 dataset
# (Replace 'sc2.csv' with the actual path if it's different)
df = pd.read_csv('sc2.csv')

# 2. Print columns to be sure (Check if it's "Solubility" or "LogS")
print("Original Columns:", df.columns.tolist())

# 3. Create a clean dataframe with ONLY [SMILES, Target]
# Note: Adjust 'Solubility' below if the column name is 'LogS' or 'Target'
clean_df = df[['SMILES', 'Solubility']]

# 4. Save to a new file for SolTrannet
clean_df.to_csv('sc2_clean_for_soltrannet.csv', index=False)
print("Created sc2_clean_for_soltrannet.csv")

In [ ]:
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score

# Load results
df = pd.read_csv('sc2_predictions.csv')

# 'true' comes from the file we just made (Col 2), 'pred' is the model output
y_true = df['true']
y_pred = df['pred']

mse = mean_squared_error(y_true, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_true, y_pred)

print("-" * 30)
print(f"SOLTRANNET SC2 RESULTS")
print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")
print("-" * 30)

## Identify the Latest Model

### Subtask:
Retrieve the filename of the most recently trained model from the `model_files` list, which corresponds to the model trained in the previous step (Cell `HuNbVMIYprBC`). This model should include '_2d_' in its name.


**Reasoning**:
I need to filter the `model_files` list to find the model that contains '_2d_' in its name and then assign it to `last_model`.



In [ ]:
last_model = '/content/sweep/aqsol_test_ind_drop0.1_ldist0.0_lattn0.5_Ndense1_heads2_dmodel8_nsl8_epochs1000_dyn8_seed420_trained.model'
print(f"Using model: {last_model}")

In [ ]:
!python predict.py --model "{last_model}" --input sc2.csv --output predictions_sc2.csv --stats --twod

In [ ]:
!python predict.py --model "{last_model}" --input test.csv --output predictions_test.csv --stats --twod